#スクレイピング   webページからデータを収集すること

In [1]:
import requests

In [ ]:
#ウェブページをダウンロード
r = requests.get('https://www.seshop.com/') #urlにアクセス
print(type(r))
print(r.status_code) #ステータスコードを取得 200だと成功

<class 'requests.models.Response'>
200


In [ ]:
text = r.text #HTMLのソースコードを取得する
for line in text.split('\n'):
    if '<title>' in line or '<h2>' in line:   #title h2 を含むかどうか
        print(line.strip())

<title>SEshop｜ 翔泳社の本・電子書籍通販サイト</title>
<h2>書籍ランキング <span class="pull-right"><a href="/ranking/1"><span class="glyphicon glyphicon-chevron-right"></span> もっと見る</a></span></h2>
<h2>電子書籍ランキング<span class="pull-right"><a href="/ranking/327/"><span class="glyphicon glyphicon-chevron-right"></span> もっと見る</a></span></h2>
<h2>予約ランキング</h2>
<h2>新刊ランキング</h2>
<h2>新刊案内</h2>
<h2>SEshop限定　紙書籍と電子書籍のセット商品</h2>
<h2>商品カテゴリー</h2>
<h2>人気商品ランキング</h2>
<h2>キャンペーン・特集</h2>


In [ ]:
#要素を抜き出す  先ほどまではHTMLを文字列として、特定の要素を探していた。今回はsoupを使うことでHTMLの構造から要素を持ってくる
from bs4 import BeautifulSoup

#HTMLを解析したBeautifulSoupオブジェクトを作成
soup = BeautifulSoup(text, 'html.parser')  #これによりHTMLを文字列でなくツリー構造で扱える
print(soup.title) #<title>タグの情報を取得
print(soup.h2) #<h2>タグの情報を取得
#h2タグの中のaタグのhref属性
print(soup.h2.a['href'])

<title>SEshop｜ 翔泳社の本・電子書籍通販サイト</title>
<h2>書籍ランキング <span class="pull-right"><a href="/ranking/1"><span class="glyphicon glyphicon-chevron-right"></span> もっと見る</a></span></h2>
/ranking/1


In [5]:
atags = soup.find_all('a')#すべてのaタグを取得
print(f'aタグの数: {len(atags)}')
for atag in atags[:5]:
    print(f'タイトル: {atag.text}')
    print(f'リンク: {atag['href']}')

aタグの数: 241
タイトル:  よくある質問
リンク: /help/detail/h337
タイトル:  法人のお客様へ
リンク: /campaign/corp
タイトル:  新規会員登録
リンク: https://www.seshop.com/regist/
タイトル:  ログイン
リンク: #modalLogin
タイトル: 
リンク: /


In [37]:
#書籍の一覧を抜き出す
from datetime import datetime

BASE_URL = 'http://www.seshop.com/product/616'
r = requests.get(BASE_URL)
soup = BeautifulSoup(r.text, 'html.parser')

books =[] #書く書籍の情報を格納するリスト
#cssセレクターで<div class='list'>  の中の
#<div class='inner'>を取得
divs = soup.select('div.list div.inner')
for div in divs:
    #画像のURLを取得
    img_url = div.find('img')['src']
    #日付のURLを取得
    day = div.find('span', class_='date').text.strip()
    day = day.removesuffix('発売')
    #日付をdatetimeに変換
    published = datetime.strptime(day, '%Y.%m.%d')

    div_txt = div.find('div', class_='txt')
    a_tag = div_txt.find('a')#aタグを取得
    title = a_tag.text.strip() #書籍タイトルを取得
    url = a_tag['href']#書籍タイトルを取得

    #販売価格を取得
    p_tags = div_txt.find_all('p')    #すべてのｐタグを取得
    price_s = p_tags[1].text          #二つ目のタグを取得（価格情報）
    price_s = price_s.split('円')[0]  #「円」で区切り、一つ目を選択
    price_s = price_s.replace(',','') #2,310 の[,]を消す
    price = int(price_s)            #整数に変換
        

    book = {
        'title': title,
        'img_url': img_url,
        'url': url,
        'price': price,
        'published': published
    }
    books.append(book)

In [38]:
books[:3]

[{'title': 'Python1年生 追加授業 もっと知りたい！プログラミングのしくみ  体験してわかる！会話でまなべる！',
  'img_url': '/static/images/product/27517/L.png',
  'url': '/product/detail/27517',
  'price': 2310,
  'published': datetime.datetime(2026, 5, 22, 0, 0)},
 {'title': 'ChatGPTプログラミング2年生 Pythonゲーム作り  体験してわかる！AIと会話でまなべる！',
  'img_url': '/static/images/product/27184/L.png',
  'url': '/product/detail/27184',
  'price': 2640,
  'published': datetime.datetime(2025, 11, 6, 0, 0)},
 {'title': 'Exercise Python  プログラマ脳を鍛える至高の問題集',
  'img_url': '/static/images/product/27150/L.png',
  'url': '/product/detail/27150',
  'price': 2508,
  'published': datetime.datetime(2025, 10, 16, 0, 0)}]

In [39]:
import pandas as pd
df = pd.DataFrame(books)
df.head(9)

,title,img_url,url,price,published
0,Python1年生 追加授業 もっと知りたい！プログラミングのしくみ 体験してわかる！会話...,/static/images/product/27517/L.png,/product/detail/27517,2310,2026-05-22
1,ChatGPTプログラミング2年生 Pythonゲーム作り 体験してわかる！AIと会話でま...,/static/images/product/27184/L.png,/product/detail/27184,2640,2025-11-06
2,Exercise Python プログラマ脳を鍛える至高の問題集,/static/images/product/27150/L.png,/product/detail/27150,2508,2025-10-16
3,Pythonレベルアップドリル 初心者から一歩進むための厳選問題集,/static/images/product/27014/L.png,/product/detail/27014,2530,2025-08-05
4,Pythonによるあたらしいデータ分析の教科書 第3版,/static/images/product/26879/L.png,/product/detail/26879,3058,2025-05-21
5,独習Python 第2版,/static/images/product/26866/L.png,/product/detail/26866,3608,2025-05-14
6,現場で使える！NumPyデータ処理入門 第2版 機械学習・データサイエンスで役立つ高速処理手法,/static/images/product/26383/L.png,/product/detail/26383,4400,2024-08-26
7,動かして学ぶ！Python Django開発入門 第3版,/static/images/product/26360/L.png,/product/detail/26360,3960,2024-08-09
8,いきなりプログラミング Python,/static/images/product/26239/L.png,/product/detail/26239,2420,2024-06-25


In [42]:
#書籍の詳細ページをクロールする
#クロールはウェブページを巡回すること
import time
from urllib import parse

#HTML上の文字列とbook辞書に保存するときのキー名の対応表
LABELS = {
    'ISBN': 'isbn',
    '判型': 'format',
    'ページ数': 'page',
}

for book in books:
    #書籍の詳細ページのURLを作成してデータを取得
    url = parse.urljoin(BASE_URL, book['url'])
    r = requests.get(url)
    soup = BeautifulSoup(r.text, 'html.parser')
    
    #書籍の詳細情報を取得
    div = soup.find('div', class_='col-md-5')

    #著者名を取得
    authors = []
    for a_tag in div.p.find_all('a'):
        authors.append(a_tag.text)
    book['author'] = authors

    #<dt>タグ、<dd>タグから情報を取得
    dl = soup.find('dl')
    dt_list = dl.find_all('dt')
    dd_list = dl.find_all('dd')
    for dt, dd in zip(dt_list, dd_list):
        dt_text = dt.text.strip()
        #<dt>タグの内容がLABELS にあれば、<dd>タグの内容をbook辞書に追加
        if dt_text in LABELS:
            dd_text = dd.text.strip()
            #ページ数の場合はint型に変換
            if dt_text == 'ページ数':
                book[LABELS[dt_text]] = int(dd_text)
            else:
                book[LABELS[dt_text]] = dd_text
    time.sleep(1)


In [44]:
books[0]

{'title': 'Python1年生 追加授業 もっと知りたい！プログラミングのしくみ  体験してわかる！会話でまなべる！',
 'img_url': '/static/images/product/27517/L.png',
 'url': '/product/detail/27517',
 'price': 2310,
 'published': datetime.datetime(2026, 5, 22, 0, 0),
 'author': ['森 巧尚'],
 'isbn': '9784798191935',
 'format': 'B5変',
 'page': 192}

In [7]:
#自然言語処理
import MeCab

In [3]:
import sys
print(sys.executable)


c:\Users\shige\anaconda3\python.exe


In [8]:
import ipadic
print(ipadic.DICDIR)

c:\Users\shige\anaconda3\Lib\site-packages\ipadic\dicdir


In [14]:
#形態素解析
text = '吾輩は猫である'
#形態素解析の結果をChaSenの出力形式で表示
t = MeCab.Tagger(f"-d {ipadic.DICDIR.replace('\\','/')}  -Ochasen")
result = t.parse(text)
print(result)

吾輩	ワガハイ	吾輩	名詞-代名詞-一般		
は	ハ	は	助詞-係助詞		
猫	ネコ	猫	名詞-一般		
で	デ	だ	助動詞	特殊・ダ	連用形
ある	アル	ある	助動詞	五段・ラ行アル	基本形
EOS



In [15]:
result

'吾輩\tワガハイ\t吾輩\t名詞-代名詞-一般\t\t\nは\tハ\tは\t助詞-係助詞\t\t\n猫\tネコ\t猫\t名詞-一般\t\t\nで\tデ\tだ\t助動詞\t特殊・ダ\t連用形\nある\tアル\tある\t助動詞\t五段・ラ行アル\t基本形\nEOS\n'

In [ ]:
#resultを改行を区切りとして行ごとに分割
results = result.splitlines() #splitlines()は'\n'で文字列を分割し、リストに格納するメソッド
#EOSの行は対象外
for res in results:
    if res == 'EOS':
        break
    res_split = res.split('\t')  #タブを区切りとして各要素に分割
    print(res_split)

['吾輩', 'ワガハイ', '吾輩', '名詞-代名詞-一般', '', '']
['は', 'ハ', 'は', '助詞-係助詞', '', '']
['猫', 'ネコ', '猫', '名詞-一般', '', '']
['で', 'デ', 'だ', '助動詞', '特殊・ダ', '連用形']
['ある', 'アル', 'ある', '助動詞', '五段・ラ行アル', '基本形']


In [23]:
#Bag of words　形態素解析の結果をもとに単語ごとの出現回数をカウントする
documents = ['子供が走る', '車が走る', '子供の脇を車が走る']

words_list = []
#形態素解析の結果をchasenの出力形式で表示
t = MeCab.Tagger(f'-d{ipadic.DICDIR.replace('\\','/')}  -Ochasen')
#各文に形態素解析を実行
for s in documents:
    s_parsed = t.parse(s) #s_parsedは形態素解析の結果　文字列
    words_s =[]
    print('s_parsed:','\n', s_parsed)
    for line in s_parsed.splitlines()[:-1]: #EOSはいらない
        words_s.append(line.split('\t')[0]) # カタカナ表記などは含めない
    words_list.append(words_s)
print(words_list)



s_parsed: 
 子供	コドモ	子供	名詞-一般		
が	ガ	が	助詞-格助詞-一般		
走る	ハシル	走る	動詞-自立	五段・ラ行	基本形
EOS

s_parsed: 
 車	クルマ	車	名詞-一般		
が	ガ	が	助詞-格助詞-一般		
走る	ハシル	走る	動詞-自立	五段・ラ行	基本形
EOS

s_parsed: 
 子供	コドモ	子供	名詞-一般		
の	ノ	の	助詞-連体化		
脇	ワキ	脇	名詞-一般		
を	ヲ	を	助詞-格助詞-一般		
車	クルマ	車	名詞-一般		
が	ガ	が	助詞-格助詞-一般		
走る	ハシル	走る	動詞-自立	五段・ラ行	基本形
EOS

[['子供', 'が', '走る'], ['車', 'が', '走る'], ['子供', 'の', '脇', 'を', '車', 'が', '走る']]


In [24]:
#行が文、列が単語の行列に各文の単語の出現回数を格納
#単語と一対一で対応する整数を保持する辞書を作る
word2int = {}
i = 0
for words in words_list:
    for word in words:    #各単語に対して処理を反復
        if word not in word2int:
            word2int[word] = i
            i += 1
print(word2int)


{'子供': 0, 'が': 1, '走る': 2, '車': 3, 'の': 4, '脇': 5, 'を': 6}


In [ ]:
#この対応関係を基にBOWを計算し文書＊単語の行列を作成
import numpy as np
#Bowを計算し行列を作成
bow = np.zeros((len(words_list), len(word2int)), dtype=int)
#各行の単語を抽出し出現回数をカウント
for i, words in enumerate(words_list):
    for word in words:
        bow[i, word2int[word]] += 1
print(bow)  #columsが単語に対応

[[1 1 1 0 0 0 0]
 [0 1 1 1 0 0 0]
 [1 1 1 1 1 1 1]]


In [28]:
#データフレームに変換し、わかりやすく
import pandas as pd
bow_df = pd.DataFrame(bow, columns=list(word2int))#リスト関数で辞書のキーのみを列名にしてる
print(list(word2int)) #リスト関数の確認
bow_df

['子供', 'が', '走る', '車', 'の', '脇', 'を']


,子供,が,走る,車,の,脇,を
0,1,1,1,0,0,0,0
1,0,1,1,1,0,0,0
2,1,1,1,1,1,1,1


In [29]:
#scikit-learnを使った計算
#単語をスペース区切りで並べた分を作成
words_split = np.array([' '.join(words) for words in words_list])
print(words_split)


['子供 が 走る' '車 が 走る' '子供 の 脇 を 車 が 走る']


In [30]:
from sklearn.feature_extraction.text import CountVectorizer
#BOWを計算
vectorizer = CountVectorizer(token_pattern='(?u)\\b\\w+\\b')
bow_vec = vectorizer.fit_transform(words_split)
bow_vec.toarray() #numpy配列に変換

array([[1, 0, 0, 1, 0, 1, 0],
       [1, 0, 0, 0, 0, 1, 1],
       [1, 1, 1, 1, 1, 1, 1]])

In [31]:
#各列が対応する単語を確認
vectorizer.get_feature_names_out()

array(['が', 'の', 'を', '子供', '脇', '走る', '車'], dtype=object)